# Computations for Table 3

Reproduce the applicability estimates for the Io stellar occultation on April 2, 2021, and the astrometric correction discussed in Section 6.

This notebook accompanies the article; equation, table and figure numbers refer
to that text. It is intended to reproduce the numerical example, so derivations
and applicability conditions are not repeated here.

**Before running:** place `light_deflection.py` and `ephemeris.py` in the notebook
directory (using these exact module names). Install NumPy, Astropy, SpiceyPy and
Jupyter; the figure notebook additionally imports Matplotlib, SciPy and tqdm.
Set `kernel` below to your local SPK path(s); the supplied absolute path is
specific to the original server. The loaded kernels must cover Earth and all
requested targets relative to the Solar System barycenter. Run cells in order
after restarting the kernel. The calls to `spice.kclear()` clear all kernels
loaded in the current process.

The stellar coordinate below is a fixed, previously propagated input; its
preparation is not rerun here. `ephem_kernel` converts reception times to TDB
and iterates geometric light time independently for each target. See its
docstring for frame, epoch, observer and returned-velocity conventions.

## Initial imports and parameters


In [1]:
import numpy as np
import astropy.units as u
from astropy.time import Time
from astropy.coordinates import SkyCoord
import spiceypy as spice

from light_deflection import delta_kpn, calc_CE, calc_der_E, calc_der_star, calc_CPE, calc_CO, calc_CT, calc_alpha_E, calc_alpha_star
from ephemeris import ephem_kernel, on_sky_velocity

In [2]:
# Occultation parameters from Morgado et al. (2022)
occ_time = Time('2021-04-02 10:24:00', scale='utc')
occ_vel = 16.54*u.km/u.s
io_radius = 1821*u.km

In [3]:
# Geocentric star position at occultation epoch from Morgado et al. (2022)
# The star position was propagated with proper motion, parallax and radial velocity
# using Butkevich & Lindegren (2014) with SORA.
# It is given here as a fixed coordinate.
pos_star = SkyCoord('21h43m04.38939019s -14d23m58.5335448s', distance=142.9253799674684*u.pc, frame='icrs').cartesian

Gravitational parameters of Jupiter and the Sun from the
[JUP365 kernel comments](https://naif.jpl.nasa.gov/pub/naif/generic_kernels/spk/satellites/jup365.cmt).

Row 1 of Table 3.


In [4]:
jup_GM = 1.266865319003704E+08 * (u.km**3) / (u.s**2)
sun_GM = 1.327132332633514E+11 * (u.km**3) / (u.s**2)
# NAIF target IDs for Jupiter, the Sun and Io.
jup_spkid = '599'
sun_spkid = '10'
io_spkid = '501'

## Ephemeris computation

Compute geocentric positions and velocities of Jupiter, the Sun and Io with
`ephem_kernel` from `ephemeris.py`. Here `observer='500'` selects the geocenter;
the helper uses SPICE Earth ID `'399'` internally.

The example uses [JUP365](https://naif.jpl.nasa.gov/pub/naif/generic_kernels/spk/satellites/jup365.bsp).
Update the local kernel path in the next cell before execution. All three
targets are evaluated at their respective iterated retarded epochs.


In [5]:
# Set this path to your local JUP365 kernel before running.
kernel = ['/srv/jupyterhub/shared/data/kernels/jup365.bsp']
spice.furnsh(kernel)

In [6]:
# Jupiter position and velocity at occultation epoch for the geocenter (observer='500')
pos_jup, vel_jup = ephem_kernel(time=occ_time, target=jup_spkid, observer='500')
pos_jup = pos_jup.cartesian

In [7]:
# Sun position and velocity at occultation epoch for the geocenter (observer='500')
pos_sun, vel_sun = ephem_kernel(time=occ_time, target=sun_spkid, observer='500')
pos_sun = pos_sun.cartesian

In [8]:
# Io position at occultation epoch for the geocenter (observer='500')
pos_io, vel_io = ephem_kernel(time=occ_time, target=io_spkid, observer='500')
pos_io_spherical = pos_io.spherical
pos_io = pos_io.cartesian

In [9]:
spice.kclear()

Observer-to-deflector distances ($D_A$) and observer-to-Io distance ($D_E$).
The two deflector distances correspond to row 2 of Table 3; $D_E$ is also printed
because it is used throughout the calculations.


In [10]:
# Distances DE and DA
DA_jup = pos_jup.norm()  # deflecting body
DA_sun = pos_sun.norm()  # deflecting body
DE_io = pos_io.norm()   # target
print('Jupiter distance (DA_jup): ', DA_jup.to(u.au))
print('Sun distance (DA_sun):', DA_sun.to(u.au))
print('Io distance (DE):', DE_io.to(u.au))

Jupiter distance (DA_jup):  5.6591845283794395 AU
Sun distance (DA_sun): 0.9996008691341834 AU
Io distance (DE): 5.6564834161105715 AU


## Angular separations

Angular separations between the deflecting body (Sun or Jupiter) and Io ($\psi_{E,A}$) and the deflecting body and the star ($\psi_{\star,A}$).

Rows 3 and 4 of Table 3.


In [11]:
psi_E_jup = SkyCoord(pos_jup).separation(SkyCoord(pos_io))
psi_star_jup = SkyCoord(pos_jup).separation(SkyCoord(pos_star))
print('Separation between Io and Jupiter (arcsec): ', psi_E_jup.arcsec)
print('Separation between the star and Jupiter (arcsec): ', psi_star_jup.arcsec)

Separation between Io and Jupiter (arcsec):  29.533925839623524
Separation between the star and Jupiter (arcsec):  29.491046723628816


In [12]:
psi_E_sun = SkyCoord(pos_sun).separation(SkyCoord(pos_io))
psi_star_sun = SkyCoord(pos_sun).separation(SkyCoord(pos_star))
print('Separation between Io and the Sun (degrees): ', psi_E_sun.deg)
print('Separation between the star and Sun (degrees): ', psi_star_sun.deg)

Separation between Io and the Sun (degrees):  49.40355845968762
Separation between the star and Sun (degrees):  49.403541522251665


## Computation of light deflections

Compute the vector light deflection using Equation 9:
$$\delta\hat{k}_{\rm pN} = \sum_A \frac{(1+\gamma)GM_A}{c^2} \frac{ \vec{r}_{SO}\times \left( \vec{r}_{AO}\times\vec{r}_{AS} \right)}{|\vec{r}_{SO}||\vec{r}_{AO}|\left(|\vec{r}_{AS}||\vec{r}_{AO}|+\vec{r}_{AS}\cdot\vec{r}_{AO}\right)}$$
and the deflection amplitudes in Equations 10 (for the star) and 12 (for the finite-distance source):
$$\alpha_{\star,A}(\psi_{\star,A}) \equiv \|\delta\hat{k}_{\star,A}\| = \frac{(1+\gamma)GM_A}{c^2D_A} \cot\left(\frac{\psi_{\star,A}}{2}\right)$$
$$\alpha_{S,A}(\psi_{S,A}) \equiv \|\delta\hat{k}_{S,A}\| =\frac{(1+\gamma)GM_A}{c^2D_A} \frac{D_S\sin\psi_{S,A}}{r_{AS} + D_A - D_S\cos\psi_{S,A}}$$

We adopted $\gamma=1$ for general relativity.

Implementation: `delta_kpn`, `calc_alpha_star` and `calc_alpha_E` in `light_deflection.py`. The vector calculation uses the supplied finite stellar distance, whereas Equation 10 uses the source-at-infinity limit; small differences are therefore expected.

The results are presented in rows 7 and 8 of Table 3


In [13]:
# Compute light deflection for the star due to Jupiter
vec_rso = - pos_star
vec_rao = - pos_jup
vec_ras = pos_star - pos_jup
print('Light Deflection due to Jupiter on the star')
# Using Equation 9
delta_k_star_jup = delta_kpn(gm=jup_GM, vec_rso=vec_rso, vec_rao=vec_rao, vec_ras=vec_ras)
print('Equation 9: ', delta_k_star_jup.norm().to(u.mas, equivalencies=u.dimensionless_angles()))
# Using Equation 10
alpha_star_jup = calc_alpha_star(gm=jup_GM, DA=DA_jup, psi_star=psi_star_jup)
print('Equation 10:', alpha_star_jup)

Light Deflection due to Jupiter on the star
Equation 9:  9.607928398587282 mas
Equation 10: 9.607930302853974 mas


In [14]:
# Compute light deflection for the star due to the Sun
vec_rso = - pos_star
vec_rao = - pos_sun
vec_ras = pos_star - pos_sun
print('Light Deflection due to the Sun on the star')
# Using Equation 9
delta_k_star_sun = delta_kpn(gm=sun_GM, vec_rso=vec_rso, vec_rao=vec_rao, vec_ras=vec_ras)
print('Equation 9: ', delta_k_star_sun.norm().to(u.mas, equivalencies=u.dimensionless_angles()))
# Using Equation 10
alpha_star_sun = calc_alpha_star(gm=sun_GM, DA=DA_sun, psi_star=psi_star_sun)
print('Equation 10:', alpha_star_sun)

Light Deflection due to the Sun on the star
Equation 9:  8.855870059212254 mas
Equation 10: 8.855870359490158 mas


In [15]:
# Compute light deflection for the ephemeris position due to Jupiter
vec_rso = - pos_io
vec_rao = - pos_jup
vec_ras = pos_io - pos_jup
print('Light Deflection due to Jupiter on the ephemeris')
# Using Equation 9
delta_k_E_jup = delta_kpn(gm=jup_GM, vec_rso=vec_rso, vec_rao=vec_rao, vec_ras=vec_ras)
print('Equation 9: ', delta_k_E_jup.norm().to(u.mas, equivalencies=u.dimensionless_angles()))
# Using Equation 12
alpha_E_jup = calc_alpha_E(gm=jup_GM, DA=DA_jup, DE=DE_io, psi_E=psi_E_jup)
print('Equation 12:', alpha_E_jup)

Light Deflection due to Jupiter on the ephemeris
Equation 9:  0.00010075760293450306 mas
Equation 12: 0.00010075760295194275 mas


In [16]:
# Compute light deflection for the ephemeris position due the Sun
vec_rso = - pos_io
vec_rao = - pos_sun
vec_ras = pos_io - pos_sun
print('Light Deflection due to the Sun on the ephemeris')
# Using Equation 9
delta_k_E_sun = delta_kpn(gm=sun_GM, vec_rso=vec_rso, vec_rao=vec_rao, vec_ras=vec_ras)
print('Equation 9: ', delta_k_E_sun.norm().to(u.mas, equivalencies=u.dimensionless_angles()))
# Using Equation 12
alpha_E_sun = calc_alpha_E(gm=sun_GM, DA=DA_sun, DE=DE_io, psi_E=psi_E_sun)
print('Equation 12:', alpha_E_sun)

Light Deflection due to the Sun on the ephemeris
Equation 9:  7.345141529713074 mas
Equation 12: 7.345141529713069 mas


## Computation of the derivatives

Here we compute the derivatives of the deflection amplitudes with respect to angular separation (Equations 39 and 40), required for the computation of the applicability estimates

$$\frac{\partial\alpha_{\star,A}}{\partial\psi_{\star,A}} = -\frac{2GM_A}{c^2D_A} \frac{1}{1-\cos\psi_{\star,A}}$$
$$\frac{\partial\alpha_{E,A}}{\partial\psi_{E,A}} = \frac{2GM_A D_E\left(D_A\cos\psi_{E,A}-D_E\right)} {c^2D_A r_{AE}\left(r_{AE}+D_A-D_E\cos\psi_{E,A}\right)}$$

Implementation: `calc_der_star` and `calc_der_E` in `light_deflection.py`. Both return signed derivatives, with angles measured in radians; the two printed values are ordered as E, then star.


In [17]:
# Derivatives where A=Jupiter
der_E_jup = calc_der_E(gm=jup_GM, DA=DA_jup, DE=DE_io, psi_E=psi_E_jup)
der_star_jup = calc_der_star(gm=jup_GM, DA=DA_jup, psi_star=psi_star_jup)
print(der_E_jup.decompose(), der_star_jup.decompose())

3.2677120872520383e-09 -0.000325791431128471


In [18]:
# Derivatives where A=Sun
der_E_sun = calc_der_E(gm=sun_GM, DA=DA_sun, DE=DE_io, psi_E=psi_E_sun)
der_star_sun = calc_der_star(gm=sun_GM, DA=DA_sun, psi_star=psi_star_sun)
print(der_E_sun.decompose(), der_star_sun.decompose())

-4.636812908166245e-08 -5.654394715380867e-08


## Computation of $C_E$

Compute the astrometric correction amplitude for Jupiter and the Sun separately, then for their vector sum (Equations 25 and 26).

$$\Delta\vec{\theta}_{E} = \delta\hat{k}_{E} -\delta\hat{k}_{\star}$$
$$C_E = D_E \left\| \Delta\vec{\theta}_{E} \right\|$$

Implementation in `light_deflection.py`: `calc_CE`.

Results are in row 9 of Table 3


In [19]:
# Compute Light deflection Correction Required for the tangent plane origin at t=t_occ due to Jupiter
CE_jup = calc_CE(DE_io, delta_k_E_jup, delta_k_star_jup)
print('CE (km) due to Jupiter: ', CE_jup.to(u.km))

CE (km) due to Jupiter:  39.41595024034985 km


In [20]:
# Compute Light deflection Correction Required for the tangent plane origin at t=t_occ
CE_sun = calc_CE(DE_io, delta_k_E_sun, delta_k_star_sun)
print('CE (km) due to the Sun: ', CE_sun.to(u.km))

CE (km) due to the Sun:  6.197738209584862 km


In [21]:
# Compute Light deflection Correction Required for the tangente plane origin at t=t_occ
CE_tot = calc_CE(DE_io, delta_k_E_jup+delta_k_E_sun, delta_k_star_jup+delta_k_star_sun)
print('CE (km) due to the combined components: ', CE_tot.to(u.km))

CE (km) due to the combined components:  45.60715636798611 km


## Computation of $C_{P-E}$

Estimate the residual from evaluating the correction at E instead of the event point P (Equation 42):
$$\widehat{C}_{P-E,A}^{\parallel}(\mathcal{R}_P) = \mathcal{R}_P \left|\frac{\partial\alpha_{E,A}}{\partial\psi_{E,A}}\right|$$
$$\widehat{C}_{P-E,A}^{\perp}(\mathcal{R}_P) = \mathcal{R}_P \frac{\alpha_{E,A}}{\sin\psi_{E,A}}$$

We choose $\mathcal{R}_P = R_{\rm Io}$.

Implementation in `light_deflection.py`: `calc_CPE`.

Results are in rows 10 and 11 of Table 3


In [22]:
CPE_jup_par, CPE_jup_per = calc_CPE(der_E=der_E_jup, RP=io_radius, alpha_E=alpha_E_jup, psi_E=psi_E_jup)
print('CPE due to Jupiter (parallel component):', CPE_jup_par.to(u.m))
print('CPE due to Jupiter (perpendicular component):', CPE_jup_per.to(u.m))

CPE due to Jupiter (parallel component): 0.005950503710885961 m
CPE due to Jupiter (perpendicular component): 0.006212502753571296 m


In [23]:
CPE_sun_par, CPE_sun_per = calc_CPE(der_E=der_E_sun, RP=io_radius, alpha_E=alpha_E_sun, psi_E=psi_E_sun)
print('CPE due to the Sun (parallel component):', CPE_sun_par.to(u.m))
print('CPE due to the Sun (perpendicular component):', CPE_sun_per.to(u.m))

CPE due to the Sun (parallel component): 0.08443636305770734 m
CPE due to the Sun (perpendicular component): 0.08540137750095578 m


## Computation of $C_{O}$

Estimate the residual from using one correction across observers (Equation 45):

$$\widehat{C}_{O,A}^{\parallel}(\mathcal{R}_O) = \mathcal{R}_O\left[\left|\frac{\partial\alpha_{E,A}}{\partial\psi_{E,A}}\right| +\frac{D_E}{D_A} \left| \frac{\partial\alpha_{\star,A}}{\partial\psi_{\star,A}} -\frac{\partial\alpha_{E,A}}{\partial\psi_{E,A}} \right|\right]$$

$$\widehat{C}_{O,A}^{\perp}(\mathcal{R}_O) = \mathcal{R}_O\left[ \frac{\alpha_{E,A}}{\sin\psi_{E,A}} +\frac{D_E}{D_A} \left| \frac{\alpha_{\star,A}}{\sin\psi_{\star,A}} -\frac{\alpha_{E,A}}{\sin\psi_{E,A}}\right|\right]$$

We choose $\mathcal{R}_O = R_{\rm Io}$.

Implementation in `light_deflection.py`: `calc_CO`.

Results are in rows 12 and 13 of Table 3


In [24]:
CO_jup_par, CO_jup_per = calc_CO(der_E=der_E_jup, der_star=der_star_jup, DA=DA_jup, DE=DE_io, RO=io_radius,
                                 alpha_E=alpha_E_jup, psi_E=psi_E_jup, alpha_star=alpha_star_jup,
                                 psi_star=psi_star_jup)
print('CO due to Jupiter (parallel component):', CO_jup_par.to(u.m))
print('CO due to Jupiter (perpendicular component):', CO_jup_per.to(u.m))

CO due to Jupiter (parallel component): 592.994930027238 m
CO due to Jupiter (perpendicular component): 592.983035937409 m


In [25]:
CO_sun_par, CO_sun_per = calc_CO(der_E=der_E_sun, der_star=der_star_sun, DA=DA_sun, DE=DE_io, RO=io_radius,
                                 alpha_E=alpha_E_sun, psi_E=psi_E_sun, alpha_star=alpha_star_sun,
                                 psi_star=psi_star_sun)
print('CO due to the Sun (parallel component):', CO_sun_par.to(u.m))
print('CO due to the Sun (perpendicular component):', CO_sun_per.to(u.m))

CO due to the Sun (parallel component): 0.18929378426741747 m
CO due to the Sun (perpendicular component): 0.18479803095516167 m


## Computation of $C_t$

Estimate the residual from using a time-independent correction throughout the event (Equation 48):

$$\widehat{C}_{t,A}^{\parallel}(\mathcal{R}_t) = \mathcal{R}_t\left[ \left|\frac{\partial\alpha_{E,A}}{\partial\psi_{E,A}}\right| +\frac{D_E W_A}{v_{\rm occ}} \left| \frac{\partial\alpha_{\star,A}}{\partial\psi_{\star,A}}-\frac{\partial\alpha_{E,A}}{\partial\psi_{E,A}}\right|\right]$$

$$\widehat{C}_{t,A}^{\perp}(\mathcal{R}_t)= \mathcal{R}_t\left[ \frac{\alpha_{E,A}}{\sin\psi_{E,A}} +\frac{D_E W_A}{v_{\rm occ}} \left| \frac{\alpha_{\star,A}}{\sin\psi_{\star,A}} -\frac{\alpha_{E,A}}{\sin\psi_{E,A}} \right|\right]$$

We choose $\mathcal{R}_t = R_{\rm Io}$.

Implementation in `light_deflection.py`: `calc_CT`.

Results are in rows 14 and 15 of Table 3


Here $W_A$ is the nonnegative on-sky angular speed computed from the two
tangential velocity components, with stellar motion neglected. It is not the
signed derivative of the star-deflector separation. The chosen track-length
scale corresponds to $\Delta t = \mathcal R_t/v_{\rm occ}$.


In [26]:
# Angular velocity between ephemeris and the star
# row 5 in Table 3
vocc_DE = (occ_vel/DE_io).to(u.arcsec/u.h, equivalencies=u.dimensionless_angles())
print('Angular velocity between ephemeris and the star', vocc_DE)

Angular velocity between ephemeris and the star 14.514136693537898 arcsec / h


In [27]:
# on_sky_velocity returns (dRA/dt * cos(Dec), dDec/dt).
# Compute angular velocities of Jupiter and Sun on sky
dot_alpha_jup, dot_delta_jup = on_sky_velocity(pos_jup, vel_jup)
dot_alpha_sun, dot_delta_sun = on_sky_velocity(pos_sun, vel_sun)
print('Jupiter on-sky velocity (RA*, DEC):', dot_alpha_jup.to(u.arcsec/u.h), dot_delta_jup.to(u.arcsec/u.h))
print('Sun on-sky velocity (RA*, DEC):', dot_alpha_sun.to(u.arcsec/u.h), dot_delta_sun.to(u.arcsec/u.h))

Jupiter on-sky velocity (RA*, DEC): 27.56233733421217 arcsec / h 9.148990858239344 arcsec / h
Sun on-sky velocity (RA*, DEC): 136.22028224243957 arcsec / h 57.62019377497182 arcsec / h


In [28]:
# Computes total angular velocities of Jupiter and Sun on sky
# row 6 in Table 3
W_jup = np.sqrt(dot_alpha_jup**2 + dot_delta_jup**2)
W_sun = np.sqrt(dot_alpha_sun**2 + dot_delta_sun**2)
print('Jupiter total on-sky velocity', W_jup.to(u.arcsec/u.h))
print('Sun total on-sky velocity', W_sun.to(u.arcsec/u.h))

Jupiter total on-sky velocity 29.04111693873108 arcsec / h
Sun total on-sky velocity 147.90555102792862 arcsec / h


In [29]:
CT_jup_par, CT_jup_per = calc_CT(der_E=der_E_jup, der_star=der_star_jup, WA=W_jup, DE=DE_io,
                                 RT=io_radius, alpha_E=alpha_E_jup, psi_E=psi_E_jup, alpha_star=alpha_star_jup,
                                 psi_star=psi_star_jup, v_occ=occ_vel)
print('CT due to Jupiter (parallel component):', CT_jup_par.to(u.m))
print('CT due to Jupiter (perpendicular component):', CT_jup_per.to(u.m))

CT due to Jupiter (parallel component): 1187.0752298804268 m
CT due to Jupiter (perpendicular component): 1187.0511573300007 m


In [30]:
CT_sun_par, CT_sun_per = calc_CT(der_E=der_E_sun, der_star=der_star_sun, WA=W_sun, DE=DE_io,
                                 RT=io_radius, alpha_E=alpha_E_sun, psi_E=psi_E_sun, alpha_star=alpha_star_sun,
                                 psi_star=psi_star_sun, v_occ=occ_vel)
print('CT due to the Sun (parallel component):', CT_sun_par.to(u.m))
print('CT due to the Sun (perpendicular component):', CT_sun_per.to(u.m))

CT due to the Sun (parallel component): 0.273267037582022 m
CT due to the Sun (perpendicular component): 0.26439812280217595 m


## Computation of Total Correction

These cells compute the total correction in $\Delta\alpha\cos\delta$ and $\Delta\delta$ to be applied to the astrometric position of Morgado et al. (2022), as discussed in Section 6.

Equations 2 and 3:
$$\hat{e}_f = (-\sin\alpha_E,\cos\alpha_E,0)$$
$$\hat{e}_g = (-\sin\delta_E\cos\alpha_E, -\sin\delta_E\sin\alpha_E, \cos\delta_E)$$

Equation 18, where $P=E$ for the tangent plane origin:
$$\Delta\vec{\theta}_{P} = - \Delta\vec{k}_{\mathrm{rel}} = \delta\hat{k}_{P} - \delta\hat{k}_{\star}$$

Project onto the tangent basis to obtain the angular components (Equation 21):
$$\Delta\alpha\cos\delta = \Delta\vec{\theta}_{E}\cdot\hat{e}_f$$
$$\Delta\delta = \Delta\vec{\theta}_{E}\cdot\hat{e}_g$$


In [31]:
e_f = SkyCoord(-np.sin(pos_io_spherical.lon), 
               np.cos(pos_io_spherical.lon), 
               0, 
               representation_type='cartesian').cartesian
e_g = SkyCoord(-np.cos(pos_io_spherical.lon)*np.sin(pos_io_spherical.lat), 
               -np.sin(pos_io_spherical.lon)*np.sin(pos_io_spherical.lat), 
               np.cos(pos_io_spherical.lat), 
               representation_type='cartesian').cartesian

In [32]:
delta_theta_jup = delta_k_E_jup - delta_k_star_jup
delta_theta_jup_f = delta_theta_jup.dot(e_f)
delta_theta_jup_g = delta_theta_jup.dot(e_g)
print('Light deflection due to Jupiter')
print('delta_RA* =', delta_theta_jup_f.to(u.mas, equivalencies=u.dimensionless_angles()))
print('delta_DEC =', delta_theta_jup_g.to(u.mas, equivalencies=u.dimensionless_angles()))

Light deflection due to Jupiter
delta_RA* = -8.870556378092303 mas
delta_DEC = -3.691013645094458 mas


In [33]:
delta_theta_sun = delta_k_E_sun - delta_k_star_sun
delta_theta_sun_f = delta_theta_sun.dot(e_f)
delta_theta_sun_g = delta_theta_sun.dot(e_g)
print('Light deflection due to the Sun')
print('delta_RA* =', delta_theta_sun_f.to(u.mas, equivalencies=u.dimensionless_angles()))
print('delta_DEC =', delta_theta_sun_g.to(u.mas, equivalencies=u.dimensionless_angles()))

Light deflection due to the Sun
delta_RA* = -1.421754026466737 mas
delta_DEC = -0.5107995480357685 mas


In [34]:
delta_theta_tot = (delta_k_E_jup + delta_k_E_sun) - (delta_k_star_jup + delta_k_star_sun)
delta_theta_tot_f = delta_theta_tot.dot(e_f)
delta_theta_tot_g = delta_theta_tot.dot(e_g)
print('Light deflection due to the combination of Jupiter and the Sun masses')
print('delta_RA* =', delta_theta_tot_f.to(u.mas, equivalencies=u.dimensionless_angles()))
print('delta_DEC =', delta_theta_tot_g.to(u.mas, equivalencies=u.dimensionless_angles()))

Light deflection due to the combination of Jupiter and the Sun masses
delta_RA* = -10.292310404559037 mas
delta_DEC = -4.201813193130228 mas
